# 02. Lazy Evaluation

transformation은 호출 시점에 실행되지 않고 계획(DAG)만 쌓인다. `explain()`으로 실행 전 계획을 확인하고, action을 두 번 호출했을 때 캐시 없이는 매번 처음부터 재계산된다는 것을 확인한다.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

spark = SparkSession.builder.appName("02_lazy_eval").getOrCreate()

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("amount", DoubleType(), False),
])
orders = spark.read.csv("/opt/spark-data/orders.csv", header=True, schema=orders_schema)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/27 01:12:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
chain = orders.filter(orders.amount > 100.0).select("user_id", "amount")
print("transformation 호출 직후 - 아직 아무 계산도 일어나지 않는다")
chain.explain()

transformation 호출 직후 - 아직 아무 계산도 일어나지 않는다
== Physical Plan ==
*(1) Filter (isnotnull(amount#2) AND (amount#2 > 100.0))
+- FileScan csv [user_id#1,amount#2] Batched: false, DataFilters: [isnotnull(amount#2), (amount#2 > 100.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/opt/spark-data/orders.csv], PartitionFilters: [], PushedFilters: [IsNotNull(amount), GreaterThan(amount,100.0)], ReadSchema: struct<user_id:int,amount:double>




In [3]:
import time

start = time.time()
first_count = chain.count()
print(f"count={first_count} elapsed={time.time() - start:.2f}s")

start = time.time()
second_count = chain.count()
print(f"count={second_count} elapsed={time.time() - start:.2f}s")

count=160431 elapsed=0.73s
count=160431 elapsed=0.16s


두 번째 `count()`가 더 빠르지만(0.73s → 0.16s) 이건 Spark 캐시 때문이 아니다 — OS 파일시스템 페이지 캐시나 JVM warmup 때문에 두 번째 읽기가 빨라질 수 있어 elapsed 시간만으로는 판단할 수 없다. 진짜 근거는 Spark UI: `.cache()` 없이는 DAG에 저장된 결과가 없으므로 `count()`마다 별도 Job이 생성되고 FileScan+Filter 스테이지를 매번 처음부터 재실행한다 — Jobs/Stages 탭에서 독립된 Job 2개로 확인 가능하다 (05_spark_ui_tour.md 참고). 04_cache_persist.ipynb에서 `.cache()`로 실제 Spark 레벨 캐싱을 확인한다.

In [4]:
spark.stop()